# Engine Condition Prediction — ANN + SMOTE with ClearML

Predicts **Engine Condition** (0 = Faulty, 1 = Good) from sensor readings.

| Step | Detail |
|------|--------|
| **Imbalance** | SMOTE applied to training set only (scaled space) |
| **Model** | Small PyTorch ANN (binary classification) |
| **Tracking** | ClearML — loss/accuracy curves, confusion matrix, artifact |


In [ ]:
from pathlib import Path
import os, pickle, copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from clearml import Logger, Task
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score,
    classification_report, f1_score,
    precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler

pd.set_option("display.max_columns", None)

RANDOM_STATE      = 42
TEST_SIZE         = 0.2
EPOCHS            = 80
BATCH_SIZE        = 128
LR                = 1e-3
PATIENCE          = 12
SMOTE_K_NEIGHBORS = 5
CLEARML_PROJECT   = "605-Engine_Condition-project"
CLEARML_TASK_NAME = "engine_condition — ANN + SMOTE"

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
data_path = Path("engine_data.csv")
print(f"Dataset path: {data_path.resolve()}")

df_raw = pd.read_csv(data_path)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

In [ ]:
print("Raw columns:", df_raw.columns.tolist())

rename_map = {
    "Engine rpm":       "engine_rpm",
    "Lub oil pressure": "lub_oil_pressure",
    "Fuel pressure":    "fuel_pressure",
    "Coolant pressure": "coolant_pressure",
    "lub oil temp":     "lub_oil_temp",
    "Coolant temp":     "coolant_temp",
    "Engine Condition": "engine_condition",
}

df = df_raw.rename(columns=rename_map).drop_duplicates().reset_index(drop=True)

TARGET_COLUMN    = "engine_condition"
NUMERIC_FEATURES = [
    "engine_rpm", "lub_oil_pressure", "fuel_pressure",
    "coolant_pressure", "lub_oil_temp", "coolant_temp",
]

missing = [c for c in NUMERIC_FEATURES + [TARGET_COLUMN] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

print("\nFinal columns:", df.columns.tolist())
df.head()

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

counts    = df[TARGET_COLUMN].value_counts()
label_map = {0: "Faulty (0)", 1: "Good (1)"}
print("\nTarget distribution:")
display(counts.rename(label_map).to_frame("count"))
print(f"\nImbalance ratio: {counts.max() / counts.min():.2f}x  → SMOTE applied to training split only.")

print("\nDescriptive statistics:")
display(df[NUMERIC_FEATURES].describe().T)

In [ ]:
target_distribution = df[TARGET_COLUMN].value_counts().sort_index()
all_classes         = sorted(df[TARGET_COLUMN].unique())
target_distribution = target_distribution.reindex(all_classes, fill_value=0)

label_map  = {0: "Faulty (0)", 1: "Good (1)"}
pie_labels = [label_map.get(c, str(c)) for c in target_distribution.index]
palette    = sns.color_palette("Set2", n_colors=len(target_distribution))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

df["_label"] = df[TARGET_COLUMN].map(label_map)
sns.countplot(data=df, x="_label", palette="Set2",
              order=[label_map[c] for c in all_classes], ax=axes[0])
axes[0].set_title("Engine Condition Distribution")
axes[0].set_xlabel("Engine Condition"); axes[0].set_ylabel("Count")
df.drop(columns=["_label"], inplace=True)

target_distribution.plot(
    kind="pie", autopct="%.1f%%", ax=axes[1],
    labels=pie_labels, colors=palette, pctdistance=0.80, startangle=90,
)
axes[1].set_ylabel(""); axes[1].set_title("Target Share")

corr = df[NUMERIC_FEATURES].corr(method="spearman")
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=axes[2])
axes[2].set_title("Spearman Correlation (features)")

plt.tight_layout(); plt.show()

In [ ]:
X = df[NUMERIC_FEATURES].values.astype(float)
y_raw = df[TARGET_COLUMN].values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = [str(c) for c in label_encoder.classes_]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Scale first, then SMOTE in scaled space
scaler          = RobustScaler()
X_train_scaled  = scaler.fit_transform(X_train)
X_test_scaled   = scaler.transform(X_test)

smote = SMOTE(k_neighbors=SMOTE_K_NEIGHBORS, random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"Train before SMOTE: { {c: int((y_train==i).sum()) for i,c in enumerate(class_names)} }")
print(f"Train after  SMOTE: { {c: int((y_train_res==i).sum()) for i,c in enumerate(class_names)} }")
print(f"Test  (untouched):  { {c: int((y_test==i).sum()) for i,c in enumerate(class_names)} }")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

before = pd.Series(y_train).value_counts().rename(label_map)
before.plot(kind="bar", ax=axes[0], color=sns.color_palette("Set2", 2), edgecolor="black", rot=0)
axes[0].set_title("Before SMOTE"); axes[0].set_ylabel("Count")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height()):,}",
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha="center", va="bottom", fontsize=11)

after = pd.Series(y_train_res).value_counts().rename(label_map)
after.plot(kind="bar", ax=axes[1], color=sns.color_palette("Set1", 2), edgecolor="black", rot=0)
axes[1].set_title("After SMOTE"); axes[1].set_ylabel("Count")
for p in axes[1].patches:
    axes[1].annotate(f"{int(p.get_height()):,}",
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha="center", va="bottom", fontsize=11)

plt.suptitle("SMOTE Effect on Training Set", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
class SmallANN(nn.Module):
    """
    Binary classifier ANN.

    Architecture:
        Input(n_features)
          → Linear(64) → BatchNorm1d → ReLU → Dropout(0.3)
          → Linear(32) → BatchNorm1d → ReLU → Dropout(0.2)
          → Linear(1)          # single logit → BCEWithLogitsLoss
    """
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)


def make_loaders(X_tr, y_tr, X_te, y_te, batch_size):
    ds_train = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32),
    )
    ds_test = TensorDataset(
        torch.tensor(X_te, dtype=torch.float32),
        torch.tensor(y_te, dtype=torch.float32),
    )
    return (DataLoader(ds_train, batch_size=batch_size, shuffle=True),
            DataLoader(ds_test,  batch_size=batch_size, shuffle=False))


model = SmallANN(n_features=len(NUMERIC_FEATURES)).to(DEVICE)
print(model)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

train_loader, test_loader = make_loaders(
    X_train_res, y_train_res.astype(np.float32),
    X_test_scaled, y_test.astype(np.float32),
    BATCH_SIZE,
)

In [ ]:
if not os.getenv("CLEARML_API_ACCESS_KEY") and not os.getenv("CLEARML_OFFLINE_MODE"):
    os.environ["CLEARML_OFFLINE_MODE"] = "1"
    print("ClearML credentials not detected; running in offline mode.")

task = Task.init(
    project_name=CLEARML_PROJECT,
    task_name=CLEARML_TASK_NAME,
    task_type=Task.TaskTypes.training,
    reuse_last_task_id=False,
)
logger = Logger.current_logger()

task.connect({
    "target_column": TARGET_COLUMN, "features": NUMERIC_FEATURES,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR, "patience": PATIENCE,
    "smote_k": SMOTE_K_NEIGHBORS, "random_state": RANDOM_STATE,
    "train_after_smote": {c: int((y_train_res==i).sum()) for i,c in enumerate(class_names)},
})
print("ClearML task initialised.")

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode="min", patience=5, factor=0.5
)

best_val_loss = float("inf")
best_weights  = copy.deepcopy(model.state_dict())
no_improve    = 0
history       = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimiser.zero_grad()
        loss = criterion(model(Xb).squeeze(1), yb)
        loss.backward()
        optimiser.step()
        running_loss += loss.item() * len(Xb)
    train_loss = running_loss / len(train_loader.dataset)

    # ── Validate ───────────────────────────────────────────────────────────────
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            out   = model(Xb).squeeze(1)
            loss  = criterion(out, yb)
            preds = (torch.sigmoid(out) >= 0.5).long()
            correct  += (preds == yb.long()).sum().item()
            val_loss += loss.item() * len(Xb)
            total    += len(Xb)
    val_loss /= total
    val_acc   = correct / total

    scheduler.step(val_loss)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    logger.report_scalar("loss", "train", train_loss, epoch)
    logger.report_scalar("loss", "val",   val_loss,   epoch)
    logger.report_scalar("accuracy", "val", val_acc,  epoch)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS}  "
              f"train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  "
              f"val_acc={val_acc:.4f}")

    # ── Early stopping ─────────────────────────────────────────────────────────
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_weights  = copy.deepcopy(model.state_dict())
        no_improve    = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stop at epoch {epoch}")
            break

model.load_state_dict(best_weights)
print("Training complete.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"],   label="Val Loss")
axes[0].set_title("Loss Curves"); axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["val_acc"], color="green", label="Val Accuracy")
axes[1].set_title("Validation Accuracy"); axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.suptitle("ANN Training History — Engine Condition", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
model.eval()
all_true, all_pred = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        out   = model(Xb.to(DEVICE)).squeeze(1)
        preds = (torch.sigmoid(out) >= 0.5).long().cpu().numpy()
        all_true.extend(yb.numpy().astype(int))
        all_pred.extend(preds)

y_true = np.array(all_true)
y_pred = np.array(all_pred)

print("ANN — Classification Report (Engine Condition)")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

metrics = {
    "test_accuracy":           float(accuracy_score(y_true, y_pred)),
    "test_f1_weighted":        float(f1_score(y_true, y_pred, average="weighted")),
    "test_f1_macro":           float(f1_score(y_true, y_pred, average="macro")),
    "test_precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
    "test_recall_weighted":    float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
}
for k, v in metrics.items():
    logger.report_scalar(f"final/{k}", "ANN", v, 0)
    print(f"  {k}: {v:.4f}")

ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=class_names, cmap="Blues"
)
plt.title("ANN — Engine Condition Confusion Matrix")
plt.tight_layout(); plt.show()

In [ ]:
output_path = Path("..") / "artifact" / "engine_condition_ann_smote.pkl"
output_path.parent.mkdir(parents=True, exist_ok=True)

bundle = {
    "model_state_dict": model.cpu().state_dict(),
    "model_class":      "SmallANN",
    "n_features":       len(NUMERIC_FEATURES),
    "scaler":           scaler,
    "label_encoder":    label_encoder,
    "target_column":    TARGET_COLUMN,
    "feature_columns":  NUMERIC_FEATURES,
    "metrics":          metrics,
    "smote_k":          SMOTE_K_NEIGHBORS,
}

with open(output_path, "wb") as f:
    pickle.dump(bundle, f)

print(f"Saved to: {output_path.resolve()}")
task.upload_artifact("engine_ann_smote", artifact_object=str(output_path.resolve()))
task.close()
print("Done.")

## Outcome

### Architecture
```
Input(6)
  → Linear(64) → BatchNorm1d → ReLU → Dropout(0.3)
  → Linear(32) → BatchNorm1d → ReLU → Dropout(0.2)
  → Linear(1)   →  BCEWithLogitsLoss
```

### Training
- Optimiser: **Adam** (`lr=1e-3`, `weight_decay=1e-4`)
- Scheduler: `ReduceLROnPlateau` (patience=5, factor=0.5)
- Early stopping: patience=12

### Class Imbalance
- **SMOTE** (`k_neighbors=5`) on training split only after `RobustScaler`